# Cell 1 — Feature D: Price Tier Classification — Model Development

## Objective

Develop and compare machine learning models that classify Amazon products into:

- Budget
- Mid-range
- Premium

The target labels were created in the preprocessing notebook using category-relative price quantiles.

## Models to Compare

1. Numeric metadata baseline
2. TF-IDF text classifier
3. Hybrid text + numeric classifier

## Leakage Prevention

The following columns must not be used as model inputs:

- `price` — used to create the target
- `price_tier` — target variable
- `search_keyword` — excluded from initial modeling to avoid category shortcut effects

Models will be evaluated using stratified train/test splits.

In [3]:
#Cell 2 — Imports
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.sparse import hstack

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)


In [4]:
#Cell 3 — Configuration
RANDOM_STATE = 42

TARGET_COLUMN = "price_tier"

NUMERIC_FEATURES = [
    "review_count",
    "average_rating",
    "review_number_collected",
    "title_length",
    "description_length",
]

TEXT_COLUMNS = [
    "title",
    "description",
    "brand",
]

CLASS_ORDER = [
    "budget",
    "mid_range",
    "premium",
]

In [5]:
"""#Cell 4A — Load Dataset (Google Colab)
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "Amazon_Marketplace_Product_Intelligence_Platform"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_d_price_tier_dataset.csv"
)

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Dataset path: {DATA_PATH}")"""

'#Cell 4A — Load Dataset (Google Colab)\nfrom google.colab import drive\n\ndrive.mount("/content/drive")\n\nPROJECT_ROOT = Path(\n    "/content/drive/MyDrive/"\n    "Amazon_Marketplace_Product_Intelligence_Platform"\n)\n\nDATA_PATH = (\n    PROJECT_ROOT\n    / "data"\n    / "processed"\n    / "feature_d_price_tier_dataset.csv"\n)\n\ndf = pd.read_csv(DATA_PATH)\n\nprint(f"Dataset shape: {df.shape}")\nprint(f"Dataset path: {DATA_PATH}")'

In [6]:
#Cell 4B — Load Dataset
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    parent
    for parent in [current_path, *current_path.parents]
    if (parent / "src").exists()
    and (parent / "data").exists()
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_d_price_tier_dataset.csv"
)

df = pd.read_csv(DATA_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset shape: {df.shape}")
print(f"Dataset path: {DATA_PATH}")

Project root: D:\AI_Trainning\Projects\Amazon_Marketplace_Product_Intelligence_Platform
Dataset shape: (946, 18)
Dataset path: D:\AI_Trainning\Projects\Amazon_Marketplace_Product_Intelligence_Platform\data\processed\feature_d_price_tier_dataset.csv


In [7]:
#Cell 5 — Validate Leakage Constraints
print("=" * 80)
print("DATASET VALIDATION AND LEAKAGE CHECK")
print("=" * 80)

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df[TARGET_COLUMN].value_counts())

print("\nChecking prohibited model features:")

PROHIBITED_FEATURES = [
    "price",
    "price_tier",
    "search_keyword",
]

for column in PROHIBITED_FEATURES:
    print(f" {column}: will NOT be used as a model feature")

assert df["price"].notna().all()
assert df[TARGET_COLUMN].notna().all()

print("\n Dataset validation passed")
print(" Leakage prevention rules established")

DATASET VALIDATION AND LEAKAGE CHECK
Dataset shape: (946, 18)

Target distribution:
price_tier
budget       327
mid_range    314
premium      305
Name: count, dtype: int64

Checking prohibited model features:
 price: will NOT be used as a model feature
 price_tier: will NOT be used as a model feature
 search_keyword: will NOT be used as a model feature

 Dataset validation passed
 Leakage prevention rules established


In [8]:

# =============================================================================
# CELL 6 — DEFINE FEATURE GROUPS
# =============================================================================

TEXT_FEATURES = [
    "title",
    "description"
]

CATEGORICAL_FEATURES = [
    "brand"
]

NUMERIC_FEATURES = [
    "review_count",
    "average_rating",
    "review_number_collected",
    "title_length",
    "description_length"
]

TARGET = "price_tier"


print("=" * 80)
print("FEATURE CONFIGURATION")
print("=" * 80)

print("\nText features:")
for feature in TEXT_FEATURES:
    print(f"  - {feature}")

print("\nCategorical features:")
for feature in CATEGORICAL_FEATURES:
    print(f"  - {feature}")

print("\nNumeric features:")
for feature in NUMERIC_FEATURES:
    print(f"  - {feature}")

print(f"\nTarget: {TARGET}")

print(
    "\nTotal model input features:",
    len(TEXT_FEATURES)
    + len(CATEGORICAL_FEATURES)
    + len(NUMERIC_FEATURES)
)

FEATURE CONFIGURATION

Text features:
  - title
  - description

Categorical features:
  - brand

Numeric features:
  - review_count
  - average_rating
  - review_number_collected
  - title_length
  - description_length

Target: price_tier

Total model input features: 8


In [9]:
# =============================================================================
# CELL 7 — TRAIN / TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split


# Separate features and target
X = df[
    TEXT_FEATURES
    + CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
].copy()

y = df[TARGET].copy()


# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("=" * 80)
print("TRAIN / TEST SPLIT")
print("=" * 80)

print(f"\nTotal samples: {len(X)}")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"Testing samples: {len(X_test)} ({len(X_test)/len(X):.1%})")


print("\nTraining target distribution:")
display(
    pd.DataFrame({
        "count": y_train.value_counts(),
        "percentage": y_train.value_counts(normalize=True).mul(100).round(2)
    })
)


print("\nTest target distribution:")
display(
    pd.DataFrame({
        "count": y_test.value_counts(),
        "percentage": y_test.value_counts(normalize=True).mul(100).round(2)
    })
)

TRAIN / TEST SPLIT

Total samples: 946
Training samples: 756 (79.9%)
Testing samples: 190 (20.1%)

Training target distribution:


,count,percentage
price_tier,,
budget,261,34.52
mid_range,251,33.20
premium,244,32.28



Test target distribution:


,count,percentage
price_tier,,
budget,66,34.74
mid_range,63,33.16
premium,61,32.11


In [10]:
# =============================================================================
# CELL 8 — EVALUATION METRICS AND CROSS-VALIDATION STRATEGY
# =============================================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# Stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# Scoring metrics for candidate comparison
SCORING = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}


# Primary model selection metric
PRIMARY_METRIC = "f1_macro"


print("=" * 80)
print("MODEL EVALUATION CONFIGURATION")
print("=" * 80)

print(f"\nCross-validation strategy: {cv}")
print(f"Number of folds: {cv.n_splits}")
print(f"Primary selection metric: {PRIMARY_METRIC}")

print("\nEvaluation metrics:")
for metric in SCORING:
    print(f"  - {metric}")

print("\nReason for primary metric:")
print(
    "Macro F1 gives equal importance to budget, mid_range, "
    "and premium classes, making it suitable for balanced "
    "multi-class price tier classification."
)

MODEL EVALUATION CONFIGURATION

Cross-validation strategy: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Number of folds: 5
Primary selection metric: f1_macro

Evaluation metrics:
  - accuracy
  - precision_macro
  - recall_macro
  - f1_macro

Reason for primary metric:
Macro F1 gives equal importance to budget, mid_range, and premium classes, making it suitable for balanced multi-class price tier classification.


In [11]:
# =============================================================================
# CELL 9 — CANDIDATE 1: NUMERIC + CATEGORICAL BASELINE PIPELINE
# =============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.linear_model import LogisticRegression


# Preprocessing for structured features
structured_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"
)


# Candidate 1 pipeline
candidate_1 = Pipeline(
    steps=[
        (
            "preprocessor",
            structured_preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


print("=" * 80)
print("CANDIDATE 1: NUMERIC + CATEGORICAL BASELINE")
print("=" * 80)

print("\nFeatures used:")
print("Numeric:")
for feature in NUMERIC_FEATURES:
    print(f"  - {feature}")

print("\nCategorical:")
for feature in CATEGORICAL_FEATURES:
    print(f"  - {feature}")

print("\nExcluded:")
print("  - title")
print("  - description")
print("  - price")
print("  - search_keyword")

print("\nModel: Logistic Regression")
print("Purpose: Establish structured-feature baseline")

CANDIDATE 1: NUMERIC + CATEGORICAL BASELINE

Features used:
Numeric:
  - review_count
  - average_rating
  - review_number_collected
  - title_length
  - description_length

Categorical:
  - brand

Excluded:
  - title
  - description
  - price
  - search_keyword

Model: Logistic Regression
Purpose: Establish structured-feature baseline


In [12]:
# =============================================================================
# CELL 10 — CROSS-VALIDATION: CANDIDATE 1
# =============================================================================

from sklearn.model_selection import cross_validate


candidate_1_cv_results = cross_validate(
    estimator=candidate_1,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)


# Create readable results table
candidate_1_results = {}

for metric in SCORING:
    scores = candidate_1_cv_results[f"test_{metric}"]

    candidate_1_results[metric] = {
        "mean": scores.mean(),
        "std": scores.std(),
        "min": scores.min(),
        "max": scores.max(),
    }


candidate_1_results_df = (
    pd.DataFrame(candidate_1_results)
    .T
    .round(4)
)

print("=" * 80)
print("CANDIDATE 1 — 5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)

display(candidate_1_results_df)

print(
    f"\nPrimary Metric ({PRIMARY_METRIC}) "
    f"Mean Score: "
    f"{candidate_1_results[PRIMARY_METRIC]['mean']:.4f}"
)

CANDIDATE 1 — 5-FOLD CROSS-VALIDATION RESULTS


,mean,std,min,max
accuracy,0.5001,0.0404,0.4276,0.5430
precision_macro,0.5099,0.0402,0.4455,0.5716
recall_macro,0.4967,0.0417,0.4213,0.5405
f1_macro,0.4884,0.0435,0.4084,0.5336



Primary Metric (f1_macro) Mean Score: 0.4884


In [13]:
# =============================================================================
# CELL 11 — CANDIDATE 2: TEXT-ONLY TF-IDF PIPELINE
# =============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


def combine_text_features(dataframe):
    """
    Combine product title, description, and brand
    into a single text representation.
    """
    combined_text = (
        dataframe["title"].fillna("").astype(str)
        + " "
        + dataframe["title"].fillna("").astype(str)
        + " "
        + dataframe["brand"].fillna("").astype(str)
        + " "
        + dataframe["description"].fillna("").astype(str)
    )

    return combined_text


# Prepare text data
X_train_text = combine_text_features(X_train)
X_test_text = combine_text_features(X_test)


# Candidate 2 pipeline
candidate_2 = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            )
        ),
    ]
)


print("=" * 80)
print("CANDIDATE 2: TEXT-ONLY TF-IDF CLASSIFIER")
print("=" * 80)

print("\nText representation:")
print("  title (repeated twice for higher importance)")
print("  + brand")
print("  + description")

print("\nTF-IDF configuration:")
print("  stop_words = english")
print("  ngram_range = (1, 2)")
print("  min_df = 2")
print("  max_df = 0.95")
print("  sublinear_tf = True")

print("\nClassifier: Logistic Regression")
print("Purpose: Evaluate predictive power of product text")

CANDIDATE 2: TEXT-ONLY TF-IDF CLASSIFIER

Text representation:
  title (repeated twice for higher importance)
  + brand
  + description

TF-IDF configuration:
  stop_words = english
  ngram_range = (1, 2)
  min_df = 2
  max_df = 0.95
  sublinear_tf = True

Classifier: Logistic Regression
Purpose: Evaluate predictive power of product text


In [14]:
# =============================================================================
# CELL 12 — CROSS-VALIDATION: CANDIDATE 2
# =============================================================================

candidate_2_cv_results = cross_validate(
    estimator=candidate_2,
    X=X_train_text,
    y=y_train,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)


# Create readable results table
candidate_2_results = {}

for metric in SCORING:
    scores = candidate_2_cv_results[f"test_{metric}"]

    candidate_2_results[metric] = {
        "mean": scores.mean(),
        "std": scores.std(),
        "min": scores.min(),
        "max": scores.max(),
    }


candidate_2_results_df = (
    pd.DataFrame(candidate_2_results)
    .T
    .round(4)
)


print("=" * 80)
print("CANDIDATE 2 — 5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)

display(candidate_2_results_df)

print(
    f"\nPrimary Metric ({PRIMARY_METRIC}) "
    f"Mean Score: "
    f"{candidate_2_results[PRIMARY_METRIC]['mean']:.4f}"
)

CANDIDATE 2 — 5-FOLD CROSS-VALIDATION RESULTS


,mean,std,min,max
accuracy,0.5066,0.0139,0.4834,0.5232
precision_macro,0.5078,0.0135,0.4882,0.5234
recall_macro,0.5055,0.0135,0.4833,0.5207
f1_macro,0.5052,0.0126,0.4853,0.5178



Primary Metric (f1_macro) Mean Score: 0.5052


In [15]:
# =============================================================================
# CELL 13 — CANDIDATE 3: HYBRID TEXT + NUMERIC + CATEGORICAL PIPELINE
# =============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


# -------------------------------------------------------------------------
# Combine title + description into a single text field.
# Brand is handled separately as a categorical feature.
# -------------------------------------------------------------------------
X_train_hybrid = X_train.copy()
X_test_hybrid = X_test.copy()

X_train_hybrid["product_text"] = (
    X_train_hybrid["title"].fillna("").astype(str)
    + " "
    + X_train_hybrid["title"].fillna("").astype(str)
    + " "
    + X_train_hybrid["description"].fillna("").astype(str)
)

X_test_hybrid["product_text"] = (
    X_test_hybrid["title"].fillna("").astype(str)
    + " "
    + X_test_hybrid["title"].fillna("").astype(str)
    + " "
    + X_test_hybrid["description"].fillna("").astype(str)
)


# -------------------------------------------------------------------------
# Hybrid preprocessing
# -------------------------------------------------------------------------
hybrid_preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
            ),
            "product_text",
        ),
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)


# -------------------------------------------------------------------------
# Candidate 3 pipeline
# -------------------------------------------------------------------------
candidate_3 = Pipeline(
    steps=[
        (
            "preprocessor",
            hybrid_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)


print("=" * 80)
print("CANDIDATE 3: HYBRID TEXT + NUMERIC + CATEGORICAL")
print("=" * 80)

print("\nText:")
print("  - title (repeated twice)")
print("  - description")

print("\nCategorical:")
print("  - brand")

print("\nNumeric:")
for feature in NUMERIC_FEATURES:
    print(f"  - {feature}")

print("\nExcluded:")
print("  - price")
print("  - search_keyword")
print("  - price_tier")

print("\nClassifier: Logistic Regression")
print("✓ All preprocessing is contained inside the pipeline")
print("✓ TF-IDF is fitted separately within each CV training fold")

CANDIDATE 3: HYBRID TEXT + NUMERIC + CATEGORICAL

Text:
  - title (repeated twice)
  - description

Categorical:
  - brand

Numeric:
  - review_count
  - average_rating
  - review_number_collected
  - title_length
  - description_length

Excluded:
  - price
  - search_keyword
  - price_tier

Classifier: Logistic Regression
✓ All preprocessing is contained inside the pipeline
✓ TF-IDF is fitted separately within each CV training fold


In [16]:
# ============================================================
# CELL 14 — Candidate 3: 5-Fold Cross-Validation
# ============================================================

from sklearn.model_selection import cross_validate

candidate_3_cv = cross_validate(
    candidate_3,
    X_train_hybrid,
    y_train,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)

candidate_3_results_df = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ],
    "mean": [
        candidate_3_cv["test_accuracy"].mean(),
        candidate_3_cv["test_precision_macro"].mean(),
        candidate_3_cv["test_recall_macro"].mean(),
        candidate_3_cv["test_f1_macro"].mean(),
    ],
    "std": [
        candidate_3_cv["test_accuracy"].std(),
        candidate_3_cv["test_precision_macro"].std(),
        candidate_3_cv["test_recall_macro"].std(),
        candidate_3_cv["test_f1_macro"].std(),
    ],
    "min": [
        candidate_3_cv["test_accuracy"].min(),
        candidate_3_cv["test_precision_macro"].min(),
        candidate_3_cv["test_recall_macro"].min(),
        candidate_3_cv["test_f1_macro"].min(),
    ],
    "max": [
        candidate_3_cv["test_accuracy"].max(),
        candidate_3_cv["test_precision_macro"].max(),
        candidate_3_cv["test_recall_macro"].max(),
        candidate_3_cv["test_f1_macro"].max(),
    ],
})

display(candidate_3_results_df)

print(
    f"\nPrimary Metric ({PRIMARY_METRIC}) "
    f"Mean Score: "
    f"{candidate_3_cv[f'test_{PRIMARY_METRIC}'].mean():.4f}"
)

,metric,mean,std,min,max
0,accuracy,0.526534,0.037627,0.467105,0.556291
1,precision_macro,0.530340,0.030769,0.479253,0.558196
2,recall_macro,0.524492,0.038788,0.462364,0.555636
3,f1_macro,0.520492,0.038749,0.456055,0.551688



Primary Metric (f1_macro) Mean Score: 0.5205


In [17]:
# =============================================================================
# CELL 15 — COMPARE ALL CANDIDATES
# =============================================================================

print("=" * 80)
print("CANDIDATE MODEL COMPARISON — 5-FOLD CROSS-VALIDATION")
print("=" * 80)


# ---------------------------------------------------------------------------
# Extract mean CV scores from each candidate
# ---------------------------------------------------------------------------

candidate_comparison_df = pd.DataFrame({
    "candidate": [
        "Candidate 1",
        "Candidate 2",
        "Candidate 3",
    ],
    
    "approach": [
        "Numeric + Categorical",
        "Text-only TF-IDF",
        "Hybrid Text + Numeric + Categorical",
    ],
    
    "accuracy": [
        candidate_1_results["accuracy"]["mean"],
        candidate_2_results["accuracy"]["mean"],
        candidate_3_cv["test_accuracy"].mean(),
    ],
    
    "precision_macro": [
        candidate_1_results["precision_macro"]["mean"],
        candidate_2_results["precision_macro"]["mean"],
        candidate_3_cv["test_precision_macro"].mean(),
    ],
    
    "recall_macro": [
        candidate_1_results["recall_macro"]["mean"],
        candidate_2_results["recall_macro"]["mean"],
        candidate_3_cv["test_recall_macro"].mean(),
    ],
    
    "f1_macro": [
        candidate_1_results["f1_macro"]["mean"],
        candidate_2_results["f1_macro"]["mean"],
        candidate_3_cv["test_f1_macro"].mean(),
    ],
})


# ---------------------------------------------------------------------------
# Rank candidates by primary metric
# ---------------------------------------------------------------------------

candidate_comparison_df = candidate_comparison_df.sort_values(
    by="f1_macro",
    ascending=False
).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Round scores for readability
# ---------------------------------------------------------------------------

display(
    candidate_comparison_df.style.format({
        "accuracy": "{:.4f}",
        "precision_macro": "{:.4f}",
        "recall_macro": "{:.4f}",
        "f1_macro": "{:.4f}",
    })
)


# ---------------------------------------------------------------------------
# Identify best candidate
# ---------------------------------------------------------------------------

best_candidate = candidate_comparison_df.iloc[0]

print("\n" + "=" * 80)
print("BEST BASELINE CANDIDATE")
print("=" * 80)

print(f"Candidate : {best_candidate['candidate']}")
print(f"Approach  : {best_candidate['approach']}")
print(f"Macro F1  : {best_candidate['f1_macro']:.4f}")


# ---------------------------------------------------------------------------
# Compare improvement over the weakest baseline
# ---------------------------------------------------------------------------

candidate_3_f1 = candidate_comparison_df.loc[
    candidate_comparison_df["candidate"] == "Candidate 3",
    "f1_macro"
].iloc[0]

candidate_2_f1 = candidate_comparison_df.loc[
    candidate_comparison_df["candidate"] == "Candidate 2",
    "f1_macro"
].iloc[0]

print(
    f"\nCandidate 3 vs Candidate 2 Macro F1 improvement: "
    f"{candidate_3_f1 - candidate_2_f1:+.4f}"
)

CANDIDATE MODEL COMPARISON — 5-FOLD CROSS-VALIDATION


,candidate,approach,accuracy,precision_macro,recall_macro,f1_macro
0,Candidate 3,Hybrid Text + Numeric + Categorical,0.5265,0.5303,0.5245,0.5205
1,Candidate 2,Text-only TF-IDF,0.5066,0.5078,0.5055,0.5052
2,Candidate 1,Numeric + Categorical,0.5001,0.5099,0.4967,0.4884



BEST BASELINE CANDIDATE
Candidate : Candidate 3
Approach  : Hybrid Text + Numeric + Categorical
Macro F1  : 0.5205

Candidate 3 vs Candidate 2 Macro F1 improvement: +0.0153


In [18]:
# =============================================================================
# CELL 16 — CANDIDATE 3: HYPERPARAMETER TUNING
# =============================================================================

from sklearn.model_selection import GridSearchCV

# ---------------------------------------------------------------------------
# Hyperparameter grid
# ---------------------------------------------------------------------------

candidate_3_param_grid = {
    "preprocessor__text__ngram_range": [
        (1, 1),
        (1, 2),
    ],
    
    "preprocessor__text__min_df": [
        1,
        2,
        3,
    ],
    
    "preprocessor__text__max_df": [
        0.90,
        0.95,
        1.00,
    ],
    
    "classifier__C": [
        0.1,
        1.0,
        10.0,
    ],
}


# ---------------------------------------------------------------------------
# GridSearchCV
# ---------------------------------------------------------------------------

candidate_3_grid_search = GridSearchCV(
    estimator=candidate_3,
    param_grid=candidate_3_param_grid,
    scoring=PRIMARY_METRIC,
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)


# ---------------------------------------------------------------------------
# Run tuning
# ---------------------------------------------------------------------------

candidate_3_grid_search.fit(
    X_train_hybrid,
    y_train,
)


# ---------------------------------------------------------------------------
# Best parameters and CV score
# ---------------------------------------------------------------------------

print("=" * 80)
print("CANDIDATE 3 — HYPERPARAMETER TUNING RESULTS")
print("=" * 80)

print("\nBest Parameters:")
for parameter, value in candidate_3_grid_search.best_params_.items():
    print(f"  {parameter}: {value}")

print(
    f"\nBest Cross-Validation {PRIMARY_METRIC}: "
    f"{candidate_3_grid_search.best_score_:.4f}"
)

CANDIDATE 3 — HYPERPARAMETER TUNING RESULTS

Best Parameters:
  classifier__C: 10.0
  preprocessor__text__max_df: 0.9
  preprocessor__text__min_df: 1
  preprocessor__text__ngram_range: (1, 1)

Best Cross-Validation f1_macro: 0.5476


In [19]:
# =============================================================================
# CELL 17 — ORIGINAL vs TUNED CANDIDATE 3
# =============================================================================

original_candidate_3_f1 = candidate_3_cv["test_f1_macro"].mean()
tuned_candidate_3_f1 = candidate_3_grid_search.best_score_

improvement = tuned_candidate_3_f1 - original_candidate_3_f1
relative_improvement = (improvement / original_candidate_3_f1) * 100


candidate_3_tuning_comparison_df = pd.DataFrame({
    "model": [
        "Original Candidate 3",
        "Tuned Candidate 3",
    ],
    "macro_f1": [
        original_candidate_3_f1,
        tuned_candidate_3_f1,
    ],
})

candidate_3_tuning_comparison_df["improvement_vs_original"] = [
    0.0,
    improvement,
]


display(
    candidate_3_tuning_comparison_df.style.format({
        "macro_f1": "{:.4f}",
        "improvement_vs_original": "{:+.4f}",
    })
)


print("\n" + "=" * 80)
print("TUNING IMPACT")
print("=" * 80)

print(f"Original Candidate 3 Macro F1 : {original_candidate_3_f1:.4f}")
print(f"Tuned Candidate 3 Macro F1    : {tuned_candidate_3_f1:.4f}")
print(f"Absolute Improvement           : {improvement:+.4f}")
print(f"Relative Improvement           : {relative_improvement:+.2f}%")

print("\nBest Hyperparameters:")
for parameter, value in candidate_3_grid_search.best_params_.items():
    print(f"  {parameter}: {value}")

,model,macro_f1,improvement_vs_original
0,Original Candidate 3,0.5205,+0.0000
1,Tuned Candidate 3,0.5476,+0.0271



TUNING IMPACT
Original Candidate 3 Macro F1 : 0.5205
Tuned Candidate 3 Macro F1    : 0.5476
Absolute Improvement           : +0.0271
Relative Improvement           : +5.20%

Best Hyperparameters:
  classifier__C: 10.0
  preprocessor__text__max_df: 0.9
  preprocessor__text__min_df: 1
  preprocessor__text__ngram_range: (1, 1)


In [20]:
# =============================================================================
# CELL 18 — FINAL TEST-SET EVALUATION: TUNED CANDIDATE 3
# =============================================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ---------------------------------------------------------------------------
# Generate predictions on the untouched test set
# ---------------------------------------------------------------------------

y_test_pred = candidate_3_grid_search.predict(X_test_hybrid)


# ---------------------------------------------------------------------------
# Calculate final test metrics
# ---------------------------------------------------------------------------

final_test_metrics = {
    "accuracy": accuracy_score(y_test, y_test_pred),
    "precision_macro": precision_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0,
    ),
    "recall_macro": recall_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0,
    ),
    "f1_macro": f1_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0,
    ),
}


# ---------------------------------------------------------------------------
# Display final metrics
# ---------------------------------------------------------------------------

final_test_results_df = pd.DataFrame({
    "metric": list(final_test_metrics.keys()),
    "test_score": list(final_test_metrics.values()),
})

display(
    final_test_results_df.style.format({
        "test_score": "{:.4f}",
    })
)


# ---------------------------------------------------------------------------
# Classification report
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL TEST-SET CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0,
    )
)


# ---------------------------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------------------------

print("=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

labels = sorted(y_test.unique())

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(
        y_test,
        y_test_pred,
        labels=labels,
    ),
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels],
)

display(confusion_matrix_df)


# ---------------------------------------------------------------------------
# Final primary metric
# ---------------------------------------------------------------------------

print(
    f"\nFinal Test {PRIMARY_METRIC}: "
    f"{final_test_metrics[PRIMARY_METRIC]:.4f}"
)

,metric,test_score
0,accuracy,0.5158
1,precision_macro,0.5274
2,recall_macro,0.5147
3,f1_macro,0.5173



FINAL TEST-SET CLASSIFICATION REPORT
              precision    recall  f1-score   support

      budget       0.64      0.55      0.59        66
   mid_range       0.43      0.54      0.48        63
     premium       0.51      0.46      0.48        61

    accuracy                           0.52       190
   macro avg       0.53      0.51      0.52       190
weighted avg       0.53      0.52      0.52       190

CONFUSION MATRIX


,Predicted_budget,Predicted_mid_range,Predicted_premium
Actual_budget,36,22,8
Actual_mid_range,10,34,19
Actual_premium,10,23,28



Final Test f1_macro: 0.5173


In [21]:
# =============================================================================
# CELL 19 — FINAL MODEL SELECTION & EVALUATION SUMMARY
# =============================================================================

# ---------------------------------------------------------------------------
# Collect all model-selection results
# ---------------------------------------------------------------------------

final_model_summary_df = pd.DataFrame({
    "stage": [
        "Candidate 1 — Baseline",
        "Candidate 2 — Text-only",
        "Candidate 3 — Hybrid",
        "Candidate 3 — Tuned",
        "Candidate 3 — Tuned Test",
    ],
    "evaluation": [
        "5-Fold CV",
        "5-Fold CV",
        "5-Fold CV",
        "5-Fold CV",
        "Held-out Test",
    ],
    "macro_f1": [
        candidate_1_results["f1_macro"]["mean"],
        candidate_2_results["f1_macro"]["mean"],
        candidate_3_cv["test_f1_macro"].mean(),
        candidate_3_grid_search.best_score_,
        final_test_metrics["f1_macro"],
    ],
})


# ---------------------------------------------------------------------------
# Display summary
# ---------------------------------------------------------------------------

display(
    final_model_summary_df.style.format({
        "macro_f1": "{:.4f}",
    })
)


# ---------------------------------------------------------------------------
# Final selected model
# ---------------------------------------------------------------------------

final_model = candidate_3_grid_search.best_estimator_

print("\n" + "=" * 80)
print("FINAL MODEL SELECTION")
print("=" * 80)

print("Selected Model : Tuned Candidate 3")
print("Approach       : Hybrid Text + Numeric + Categorical")
print(f"CV Macro F1    : {candidate_3_grid_search.best_score_:.4f}")
print(f"Test Macro F1  : {final_test_metrics['f1_macro']:.4f}")

print("\nBest Hyperparameters:")
for parameter, value in candidate_3_grid_search.best_params_.items():
    print(f"  {parameter}: {value}")

,stage,evaluation,macro_f1
0,Candidate 1 — Baseline,5-Fold CV,0.4884
1,Candidate 2 — Text-only,5-Fold CV,0.5052
2,Candidate 3 — Hybrid,5-Fold CV,0.5205
3,Candidate 3 — Tuned,5-Fold CV,0.5476
4,Candidate 3 — Tuned Test,Held-out Test,0.5173



FINAL MODEL SELECTION
Selected Model : Tuned Candidate 3
Approach       : Hybrid Text + Numeric + Categorical
CV Macro F1    : 0.5476
Test Macro F1  : 0.5173

Best Hyperparameters:
  classifier__C: 10.0
  preprocessor__text__max_df: 0.9
  preprocessor__text__min_df: 1
  preprocessor__text__ngram_range: (1, 1)


In [22]:
"""# =============================================================================
# CELL 20a — SAVE FINAL FEATURE D MODEL AND METADATA (GOOGLE COLAB)
# =============================================================================

from pathlib import Path
import json
import joblib
from datetime import datetime, timezone

# -------------------------------------------------------------------------
# Output directory
# -------------------------------------------------------------------------
MODEL_DIR = Path("artifacts/feature_d_price_tier")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "price_tier_model.joblib"
METADATA_PATH = MODEL_DIR / "price_tier_metadata.json"

# -------------------------------------------------------------------------
# Save trained model
# -------------------------------------------------------------------------
joblib.dump(
    final_model,
    MODEL_PATH,
)

# -------------------------------------------------------------------------
# Build metadata
# -------------------------------------------------------------------------
feature_d_metadata = {
    "feature": "price_tier_classification",
    "model_type": "hybrid_text_numeric_categorical_logistic_regression",

    "target": "price_tier",
    "target_classes": sorted(y_train.unique().tolist()),

    "dataset_rows": int(len(X)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),

    "model_features": {
        "text_features": [
            "title",
            "description",
        ],
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
    },

    "prohibited_features": [
        "price",
        "price_tier",
        "search_keyword",
    ],

    "text_representation": (
        "title repeated twice + description"
    ),

    "best_hyperparameters": {
        parameter: value
        for parameter, value
        in candidate_3_grid_search.best_params_.items()
    },

    "evaluation": {
        "primary_metric": PRIMARY_METRIC,
        "cv_macro_f1": float(candidate_3_grid_search.best_score_),
        "test_accuracy": float(final_test_metrics["accuracy"]),
        "test_precision_macro": float(final_test_metrics["precision_macro"]),
        "test_recall_macro": float(final_test_metrics["recall_macro"]),
        "test_f1_macro": float(final_test_metrics["f1_macro"]),
    },

    "random_state": 42,

    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

# -------------------------------------------------------------------------
# Save metadata
# -------------------------------------------------------------------------
with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        feature_d_metadata,
        file,
        indent=4,
    )

# -------------------------------------------------------------------------
# Verification
# -------------------------------------------------------------------------
print("=" * 80)
print("FEATURE D — FINAL MODEL SAVED")
print("=" * 80)

print(f"Model path   : {MODEL_PATH}")
print(f"Metadata path: {METADATA_PATH}")

print("\nModel exists   :", MODEL_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

print("\nFinal Model:")
print("  Approach    : Hybrid Text + Numeric + Categorical")
print(f"  CV Macro F1 : {candidate_3_grid_search.best_score_:.4f}")
print(f"  Test Macro F1: {final_test_metrics['f1_macro']:.4f}")

print("\nSaved successfully.")"""

'# =============================================================================\n# CELL 20a — SAVE FINAL FEATURE D MODEL AND METADATA (GOOGLE COLAB)\n# =============================================================================\n\nfrom pathlib import Path\nimport json\nimport joblib\nfrom datetime import datetime, timezone\n\n# -------------------------------------------------------------------------\n# Output directory\n# -------------------------------------------------------------------------\nMODEL_DIR = Path("artifacts/feature_d_price_tier")\nMODEL_DIR.mkdir(parents=True, exist_ok=True)\n\nMODEL_PATH = MODEL_DIR / "price_tier_model.joblib"\nMETADATA_PATH = MODEL_DIR / "price_tier_metadata.json"\n\n# -------------------------------------------------------------------------\n# Save trained model\n# -------------------------------------------------------------------------\njoblib.dump(\n    final_model,\n    MODEL_PATH,\n)\n\n# -----------------------------------------------------

In [24]:
# =============================================================================
# CELL 20b — SAVE FINAL FEATURE D MODEL AND METADATA 
# =============================================================================

from pathlib import Path
import json
import joblib
from datetime import datetime, timezone

# -------------------------------------------------------------------------
# Project-relative output directory
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent.parent

MODEL_DIR = PROJECT_ROOT / "models" / "price_tier"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "price_tier_model.joblib"
METADATA_PATH = MODEL_DIR / "price_tier_metadata.json"

# -------------------------------------------------------------------------
# Save trained model
# -------------------------------------------------------------------------
joblib.dump(
    final_model,
    MODEL_PATH,
)

# -------------------------------------------------------------------------
# Build metadata
# -------------------------------------------------------------------------
feature_d_metadata = {
    "feature": "price_tier_classification",
    "model_type": "hybrid_text_numeric_categorical_logistic_regression",

    "target": "price_tier",
    "target_classes": sorted(y_train.unique().tolist()),

    "dataset_rows": int(len(X)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),

    "model_features": {
        "text_features": [
            "title",
            "description",
        ],
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
    },

    "prohibited_features": [
        "price",
        "price_tier",
        "search_keyword",
    ],

    "text_representation": (
        "title repeated twice + description"
    ),

    "best_hyperparameters": {
        parameter: value
        for parameter, value
        in candidate_3_grid_search.best_params_.items()
    },

    "evaluation": {
        "primary_metric": PRIMARY_METRIC,
        "cv_macro_f1": float(candidate_3_grid_search.best_score_),
        "test_accuracy": float(final_test_metrics["accuracy"]),
        "test_precision_macro": float(final_test_metrics["precision_macro"]),
        "test_recall_macro": float(final_test_metrics["recall_macro"]),
        "test_f1_macro": float(final_test_metrics["f1_macro"]),
    },

    "random_state": 42,

    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

# -------------------------------------------------------------------------
# Save metadata
# -------------------------------------------------------------------------
with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        feature_d_metadata,
        file,
        indent=4,
    )

# -------------------------------------------------------------------------
# Verification
# -------------------------------------------------------------------------
print("=" * 80)
print("FEATURE D — FINAL MODEL SAVED")
print("=" * 80)

print(f"Model path   : {MODEL_PATH}")
print(f"Metadata path: {METADATA_PATH}")

print("\nModel exists   :", MODEL_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

print("\nFinal Model:")
print("  Approach     : Hybrid Text + Numeric + Categorical")
print(f"  CV Macro F1  : {candidate_3_grid_search.best_score_:.4f}")
print(f"  Test Macro F1: {final_test_metrics['f1_macro']:.4f}")

print("\nSaved successfully.")

FEATURE D — FINAL MODEL SAVED
Model path   : d:\AI_Trainning\Projects\Amazon_Marketplace_Product_Intelligence_Platform\models\price_tier\price_tier_model.joblib
Metadata path: d:\AI_Trainning\Projects\Amazon_Marketplace_Product_Intelligence_Platform\models\price_tier\price_tier_metadata.json

Model exists   : True
Metadata exists: True

Final Model:
  Approach     : Hybrid Text + Numeric + Categorical
  CV Macro F1  : 0.5476
  Test Macro F1: 0.5173

Saved successfully.


In [25]:
# =============================================================================
# CELL 21 — RELOAD SAVED FEATURE D MODEL AND VERIFY PREDICTIONS
# =============================================================================

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------
MODEL_PATH = MODEL_DIR / "price_tier_model.joblib"
METADATA_PATH = MODEL_DIR / "price_tier_metadata.json"

# -------------------------------------------------------------------------
# Reload model
# -------------------------------------------------------------------------
reloaded_model = joblib.load(MODEL_PATH)

# -------------------------------------------------------------------------
# Reload metadata
# -------------------------------------------------------------------------
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    reloaded_metadata = json.load(file)

# -------------------------------------------------------------------------
# Verify model and metadata
# -------------------------------------------------------------------------
print("=" * 80)
print("FEATURE D — SAVED MODEL RELOAD VERIFICATION")
print("=" * 80)

print("\nModel file exists    :", MODEL_PATH.exists())
print("Metadata file exists :", METADATA_PATH.exists())

print("\nModel type:")
print(f"  {type(reloaded_model).__name__}")

print("\nMetadata:")
print(f"  Feature        : {reloaded_metadata['feature']}")
print(f"  Target         : {reloaded_metadata['target']}")
print(f"  Target classes : {reloaded_metadata['target_classes']}")

print("\nEvaluation:")
print(
    f"  CV Macro F1    : "
    f"{reloaded_metadata['evaluation']['cv_macro_f1']:.4f}"
)
print(
    f"  Test Macro F1  : "
    f"{reloaded_metadata['evaluation']['test_f1_macro']:.4f}"
)

# -------------------------------------------------------------------------
# Generate predictions using the reloaded model
# -------------------------------------------------------------------------
reloaded_predictions = reloaded_model.predict(X_test_hybrid)

print("\nPrediction verification:")
print(f"  Test samples       : {len(X_test_hybrid)}")
print(f"  Predictions        : {len(reloaded_predictions)}")
print(
    f"  Valid predictions  : "
    f"{set(reloaded_predictions).issubset(set(reloaded_metadata['target_classes']))}"
)

# -------------------------------------------------------------------------
# Compare reloaded predictions with original predictions
# -------------------------------------------------------------------------
predictions_match = np.array_equal(
    y_test_pred,
    reloaded_predictions,
)

print(
    f"  Match original predictions : {predictions_match}"
)

# -------------------------------------------------------------------------
# Display sample predictions
# -------------------------------------------------------------------------
verification_df = X_test_hybrid[
    ["title", "brand"]
].copy()

verification_df["actual_price_tier"] = y_test.values
verification_df["predicted_price_tier"] = reloaded_predictions

display(
    verification_df.head(10)
)

# -------------------------------------------------------------------------
# Final verification status
# -------------------------------------------------------------------------
verification_passed = (
    MODEL_PATH.exists()
    and METADATA_PATH.exists()
    and predictions_match
    and len(reloaded_predictions) == len(y_test)
    and set(reloaded_predictions).issubset(
        set(reloaded_metadata["target_classes"])
    )
)

print("\n" + "=" * 80)

if verification_passed:
    print("FEATURE D MODEL VERIFICATION: PASSED")
    print("=" * 80)
    print(
        "The saved model was successfully reloaded and "
        "produced identical predictions."
    )
else:
    print("FEATURE D MODEL VERIFICATION: FAILED")
    print("=" * 80)
    print(
        "Please inspect the model, metadata, and prediction comparison."
    )

FEATURE D — SAVED MODEL RELOAD VERIFICATION

Model file exists    : True
Metadata file exists : True

Model type:
  Pipeline

Metadata:
  Feature        : price_tier_classification
  Target         : price_tier
  Target classes : ['budget', 'mid_range', 'premium']

Evaluation:
  CV Macro F1    : 0.5476
  Test Macro F1  : 0.5173

Prediction verification:
  Test samples       : 190
  Predictions        : 190
  Valid predictions  : True
  Match original predictions : True


,title,brand,actual_price_tier,predicted_price_tier
150,Logitech MX Brio Ultra HD 4K Webcam Collaborat...,No Brand,premium,premium
629,COOFANDY Mens Knit Polo Shirts Short Sleeve Ri...,COOFANDY,premium,mid_range
940,"38inch Wood Acoustic Guitar for Adults, 6 Stee...",Brand: Abundant financial resources,mid_range,mid_range
683,"TIEFOSSI Line Journal Notebook for Women Men, ...",TIEFOSSI,premium,budget
561,"YETI Rambler 36 oz Bottle, Vacuum Insulated, L...",YETI,premium,mid_range
248,COMFEE',COMFEE',budget,mid_range
26,TAGRY Hybrid Active Noise Cancelling Wireless ...,TAGRY,premium,mid_range
515,JanSport Laptop Backpack - Computer Bag with 2...,JanSport,premium,premium
618,"Gildan Adult Ultra Cotton T-Shirt, Style G2000...",Gildan,mid_range,mid_range
311,"Lenovo 15.6"" Business Laptop, 2026 Edition, 8G...",Lenovo,premium,mid_range



FEATURE D MODEL VERIFICATION: PASSED
The saved model was successfully reloaded and produced identical predictions.


# Feature D — Final Interpretation and Limitations

## 1. Final Model

The selected model is **Tuned Candidate 3 — Hybrid Text + Numeric + Categorical Logistic Regression**.

The model combines:

- Product title and description using TF-IDF
- Numeric product attributes
- Brand as a categorical feature

The target variable is `price_tier` with three classes:

- `budget`
- `mid_range`
- `premium`

---

## 2. Leakage Prevention

The following fields were explicitly excluded from the model features:

- `price`
- `price_tier`
- `search_keyword`

`price` was used only for target construction and exploratory analysis.

Therefore, the model does not directly receive the product price when predicting the price tier.

---

## 3. Model Selection

Three candidate approaches were compared using 5-fold cross-validation with **Macro F1** as the primary metric.

| Candidate | Approach | Macro F1 |
|---|---|---:|
| Candidate 1 | Numeric + Categorical | 0.4884 |
| Candidate 2 | Text-only TF-IDF | 0.5052 |
| Candidate 3 | Hybrid Text + Numeric + Categorical | 0.5205 |
| Tuned Candidate 3 | Tuned Hybrid Model | **0.5476** |

The hybrid Candidate 3 performed better than the simpler baseline approaches.

Hyperparameter tuning further improved the cross-validation Macro F1 from **0.5205 to 0.5476**.

---

## 4. Final Held-Out Test Performance

The tuned model was evaluated once on the held-out test set.

| Metric | Test Score |
|---|---:|
| Accuracy | 0.5158 |
| Precision (Macro) | 0.5274 |
| Recall (Macro) | 0.5147 |
| **F1 (Macro)** | **0.5173** |

The held-out test Macro F1 of **0.5173** is the final unbiased estimate of model performance on unseen data.

---

## 5. Class-Level Performance

| Price Tier | Precision | Recall | F1 | Support |
|---|---:|---:|---:|---:|
| Budget | 0.64 | 0.55 | **0.59** | 66 |
| Mid-range | 0.43 | 0.54 | 0.48 | 63 |
| Premium | 0.51 | 0.46 | 0.48 | 61 |

The **budget** class was predicted most effectively.

The **mid-range** and **premium** classes were more difficult to distinguish from each other.

---

## 6. Interpretation

The results show that product text, brand, and other product attributes contain some information about relative price positioning.

However, the classification problem is inherently challenging because the actual `price` variable is intentionally excluded from the model.

In addition, different product types can have very different normal price ranges. The `search_keyword` field was also excluded so that the model would not rely on search-context information.

Therefore, the model must infer price tier from observable product characteristics rather than directly using price or product-category context.

---

## 7. Limitations

The final held-out test Macro F1 of approximately **0.52** indicates moderate classification performance rather than highly reliable price-tier prediction.

The model should therefore be treated as an **estimated price-tier classifier**, not as an exact pricing or valuation system.

Further improvement could potentially come from richer product attributes, larger datasets, better representation of product specifications, or additional modeling approaches.

---

## 8. Final Status

**Feature D model development: COMPLETE**

- Model comparison: Complete
- Hyperparameter tuning: Complete
- Held-out test evaluation: Complete
- Leakage checks: Complete
- Final model selection: Complete
- Final model artifact: Saved
- Metadata: Saved
- Model reload verification: Completed